# Import SSB municipality population

Downloads municipality population from Statistics Norway's Statbank API table 06913 and writes a small Parquet dimension table.

The join key is `kommunenummer`, which is already present in the company register as `forretningsadresse.kommunenummer`. 

Source: SSB table 06913, Population and population changes, by region, contents and year. The selected year and extraction timestamp are stored with the output.

In [1]:
import itertools
import json
import os
from datetime import datetime, timezone

import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from staged_write import write_staged

DATA_DIR = "/home/jovyan/data"
SSB_TABLE = "06913"
POPULATION_YEAR = "2026"
OUTPUT_PATH = os.path.join(
    DATA_DIR,
    "parquet",
    f"ssb_population_{POPULATION_YEAR}.parquet",
)
SSB_URL = f"https://data.ssb.no/api/v0/en/table/{SSB_TABLE}"

spark = SparkSession.builder.appName("group13_import_ssb_population").getOrCreate()
print("Spark          ", spark.version)
print("SSB table      ", SSB_TABLE)
print("Population year", POPULATION_YEAR)
print("Output         ", OUTPUT_PATH)

Spark           4.2.0
SSB table       06913
Population year 2026
Output          /home/jovyan/data/parquet/ssb_population_2026.parquet


## Select current municipalities

SSB's region dimension includes counties, the whole country, historical municipality codes, and special regions. Current Norwegian municipality codes are four digits and do not have a suffix such as `u`. The year filter below removes historical municipality codes that have no population value for the selected year.

In [2]:
metadata_response = requests.get(SSB_URL, timeout=60)
metadata_response.raise_for_status()
table_metadata = metadata_response.json()

region_variable = next(
    variable for variable in table_metadata["variables"]
    if variable["code"] == "Region"
)

region_codes = [
    code
    for code in region_variable["values"]
    if len(code) == 4 and code.isdigit()
]

assert region_codes, "SSB metadata returned no four-digit municipality candidates"
print("Municipality candidates in metadata:", len(region_codes))

Municipality candidates in metadata: 1184


In [3]:
query = {
    "query": [
        {
            "code": "Region",
            "selection": {
                "filter": "item",
                "values": region_codes,
            },
        },
        {
            "code": "ContentsCode",
            "selection": {
                "filter": "item",
                "values": ["Folkemengde"],
            },
        },
        {
            "code": "Tid",
            "selection": {
                "filter": "item",
                "values": [POPULATION_YEAR],
            },
        },
    ],
    "response": {
        "format": "json-stat2",
    },
}

data_response = requests.post(SSB_URL, json=query, timeout=60)
data_response.raise_for_status()
ssb_payload = data_response.json()
print("Returned dimensions:", ssb_payload["id"])
print("Returned values:", len(ssb_payload["value"]))

Returned dimensions: ['Region', 'ContentsCode', 'Tid']
Returned values: 1184


## Convert JSON-stat2 to rows

JSON-stat2 stores dimension labels and observations separately. The conversion keeps the municipality code as a string so leading zeroes, such as `0301` for Oslo, are preserved.

In [4]:
def ordered_category_codes(dimension):
    category = dimension["category"]
    index = category["index"]
    if isinstance(index, dict):
        return [code for code, _ in sorted(index.items(), key=lambda item: item[1])]
    return list(index)


dimension_order = ssb_payload["id"]
dimension_codes = [
    ordered_category_codes(ssb_payload["dimension"][dimension_name])
    for dimension_name in dimension_order
]

observations = []
for combination, value in zip(
    itertools.product(*dimension_codes),
    ssb_payload["value"],
):
    row = dict(zip(dimension_order, combination))
    row["population"] = value
    observations.append(row)

population_rows = [
    {
        "kommunenummer": row["Region"].zfill(4),
        "population": int(row["population"]),
        "population_year": int(POPULATION_YEAR),
        "source_table": SSB_TABLE,
        "source_retrieved_at": datetime.now(timezone.utc).isoformat(),
    }
    for row in observations
    if row["population"] is not None
]

assert population_rows, "SSB returned no non-null municipality observations"
assert len({row["kommunenummer"] for row in population_rows}) == len(population_rows)
assert all(len(row["kommunenummer"]) == 4 for row in population_rows)

print("Municipality rows with population:", len(population_rows))
print("Population range:", min(row["population"] for row in population_rows), "to", max(row["population"] for row in population_rows))

Municipality rows with population: 1184
Population range: 0 to 728714


## Write and verify the Parquet dimension

This file is intentionally separate from the existing company and financial mirrors. The analytics build can join it by `kommunenummer` without changing the benchmark inputs.

In [5]:
population_df = spark.createDataFrame(population_rows)

write_staged(population_df, OUTPUT_PATH, "parquet")

written = spark.read.parquet(OUTPUT_PATH)
written_count = written.count()
distinct_count = written.select("kommunenummer").distinct().count()

assert written_count == len(population_rows)
assert distinct_count == written_count
assert written.filter(F.col("population").isNull()).count() == 0

written.orderBy("kommunenummer").show(10, truncate=False)
print("Wrote", written_count, "municipality rows to", OUTPUT_PATH)

  staged write: 10 files, 0.00 GB copied to /home/jovyan/data/parquet/ssb_population_2026.parquet


+-------------+----------+---------------+--------------------------------+------------+
|kommunenummer|population|population_year|source_retrieved_at             |source_table|
+-------------+----------+---------------+--------------------------------+------------+
|0101         |0         |2026           |2026-09-11T21:17:07.804812+00:00|06913       |
|0102         |0         |2026           |2026-09-11T21:17:07.804813+00:00|06913       |
|0103         |0         |2026           |2026-09-11T21:17:07.804815+00:00|06913       |
|0104         |0         |2026           |2026-09-11T21:17:07.804816+00:00|06913       |
|0105         |0         |2026           |2026-09-11T21:17:07.804817+00:00|06913       |
|0106         |0         |2026           |2026-09-11T21:17:07.804818+00:00|06913       |
|0111         |0         |2026           |2026-09-11T21:17:07.804820+00:00|06913       |
|0112         |0         |2026           |2026-09-11T21:17:07.804821+00:00|06913       |
|0113         |0     